# Import Library and set up path

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
BASE_DIR = Path.cwd().parent
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

# Create processed folder if it doesn't exist
DATA_PROCESSED.mkdir(exist_ok=True)

print("Raw Data:", DATA_RAW)
print("Processed Data:", DATA_PROCESSED)

Raw Data: C:\Users\gaura\Desktop\bluestock_mf_capstone\data\raw
Processed Data: C:\Users\gaura\Desktop\bluestock_mf_capstone\data\processed


In [4]:
print(pd.read_csv("../data/raw/02_nav_history.csv").columns.tolist())
print(pd.read_csv("../data/raw/08_investor_transactions.csv").columns.tolist())
print(pd.read_csv("../data/raw/07_scheme_performance.csv").columns.tolist())

['amfi_code', 'date', 'nav']
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


# Clean nav_history.csv

In [6]:
# Load Data
nav = pd.read_csv(DATA_RAW / "02_nav_history.csv")

print(nav.shape)
nav.head()

(46000, 3)


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [8]:
# Convert date column

nav["date"] = pd.to_datetime(
    nav["date"],
    errors="coerce"
)

print(nav.dtypes)

amfi_code             int64
date         datetime64[ns]
nav                 float64
dtype: object


In [10]:
# Sort data

nav = nav.sort_values(
    ["amfi_code", "date"]
)

In [12]:
# Remove duplicates

before = len(nav)

nav = nav.drop_duplicates()

after = len(nav)

print("Duplicates Removed:", before - after)

Duplicates Removed: 0


In [14]:
# Forward fill NAV

nav["nav"] = (
    nav.groupby("amfi_code")["nav"]
    .ffill()
)

In [16]:
# Validate NAV

invalid_nav = nav[
    nav["nav"] <= 0
]

print("Invalid NAV Rows:", len(invalid_nav))

Invalid NAV Rows: 0


In [18]:
nav.to_csv(
    DATA_PROCESSED / "nav_history_clean.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


# Clean investor_transaction.csv

In [24]:
# Load Data
txn = pd.read_csv(
    DATA_RAW / "08_investor_transactions.csv"
)

print(txn.shape)
txn.head()

(32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [26]:
# Convert date column
txn["transaction_date"] = pd.to_datetime(
    txn["transaction_date"],
    errors="coerce"
)

In [30]:
# Standardized Transaction Type
txn["transaction_type"] = (
    txn["transaction_type"]
    .str.strip()
    .str.title()
)

print(txn["transaction_type"].value_counts())

transaction_type
Sip           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64


In [32]:
# Validate > 0
invalid_amounts = txn[
    txn["amount_inr"] <= 0
]

print("Invalid Amount Rows:", len(invalid_amounts))

Invalid Amount Rows: 0


In [34]:
# Check KYC Status
print(
    txn["kyc_status"]
    .value_counts(dropna=False)
)

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


In [36]:
# Save Clean File
txn.to_csv(
    DATA_PROCESSED / "transactions_clean.csv",
    index=False
)

print("Transactions Saved")

Transactions Saved


# Clean Scheme_performance.csv

In [41]:
# Load Data
perf = pd.read_csv(
    DATA_RAW / "07_scheme_performance.csv"
)

print(perf.shape)
perf.head()

(40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [43]:
# Check Data Type
perf.dtypes

amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object

In [47]:
# Find Negative Sharpe Ratio
negative_sharpe = perf[
    perf["sharpe_ratio"] < 0
]

print(
    "Negative Sharpe Funds:",
    len(negative_sharpe)
)

Negative Sharpe Funds: 0


In [51]:
# Validate Expense Ratio Range(0.1% to 2.5%)
expense_issues = perf[
    (perf["expense_ratio_pct"] < 0.1)
    |
    (perf["expense_ratio_pct"] > 2.5)
]

print(
    "Expense Ratio Issues:",
    len(expense_issues)
)

Expense Ratio Issues: 0


In [53]:
# Save Clean File
perf.to_csv(
    DATA_PROCESSED / "scheme_performance_clean.csv",
    index=False
)

print("Performance File Saved")

Performance File Saved


In [55]:
pip show sqlalchemy

Name: SQLAlchemyNote: you may need to restart the kernel to use updated packages.

Version: 2.0.25
Summary: Database Abstraction Library
Home-page: https://www.sqlalchemy.org
Author: Mike Bayer
Author-email: mike_mp@zzzcomputing.com
License: MIT
Location: C:\Users\public\anaconda3\Lib\site-packages
Requires: greenlet, typing-extensions
Required-by: 


In [1]:
import sqlalchemy
print(sqlalchemy.__version__)

2.0.25


In [7]:
# Database folder setup
from sqlalchemy import create_engine

DB_DIR = BASE_DIR / "data" / "db"
DB_DIR.mkdir(exist_ok=True)

print(DB_DIR)

C:\Users\gaura\Desktop\bluestock_mf_capstone\data\db


In [9]:
#  Create Database
engine = create_engine(
    f"sqlite:///{DB_DIR}/bluestock_mf.db"
)

print("Database Created Successfully")

Database Created Successfully


In [13]:
# Loading Clean Files
nav = pd.read_csv(
    DATA_PROCESSED / "nav_history_clean.csv"
)

txn = pd.read_csv(
    DATA_PROCESSED / "transactions_clean.csv"
)

perf = pd.read_csv(
    DATA_PROCESSED / "scheme_performance_clean.csv"
)

print(nav.shape)
print(txn.shape)
print(perf.shape)

(46000, 3)
(32778, 13)
(40, 19)


In [15]:
# Loading in SQLite
nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

txn.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

perf.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

print("All Tables Loaded Successfully")

All Tables Loaded Successfully


In [21]:
# Verify Table
pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    engine
)

,name
0,fact_nav
1,fact_transactions
2,fact_performance


In [23]:
# Row Count Verification
print(
    pd.read_sql(
        "SELECT COUNT(*) AS rows FROM fact_nav",
        engine
    )
)

print(
    pd.read_sql(
        "SELECT COUNT(*) AS rows FROM fact_transactions",
        engine
    )
)

print(
    pd.read_sql(
        "SELECT COUNT(*) AS rows FROM fact_performance",
        engine
    )
)

    rows
0  46000
    rows
0  32778
   rows
0    40
